# Final project ver2.0 — 파인튜닝 추가 데이터

    **Final project ver2.0** — Windows + VS Code + Colab GPU.

    로컬 파일과 Colab 서버는 서로 다른 컴퓨터입니다. VS Code에서 `Select Kernel → Colab`을 선택하고 GPU를 확인하세요. 처음에는 `RUN_CONDITIONS=["D"]`로 기반을 확인합니다. **A·B·D는 제공하는 완성 코드**입니다. A/B에는 저장 VLA 모델 경로만 지정하면 됩니다. C는 학생의 루프를 연결합니다. 기존 `Final_project.ipynb`와 연구 조건의 의미가 다릅니다.

    씬·시작 명령·내부 프롬프트·루프 입출력의 자세한 순서는 **`docs/FINAL_PROJECT_V2_STUDENT_GUIDE.md`**를 먼저 읽으세요. 내부 프롬프트는 `final_project_v2/prompts/starter_strategy.txt`, C 루프는 `final_project_v2/student_strategy.py`입니다. B와 C는 기본적으로 같은 TXT를 읽습니다.

    기본 생성 씬은 시작 예시입니다. 기하 검사 결과는 실제 파지 불가능의 정답이 아닙니다. 실제 실행 영상으로 확인하세요. 안내는 `docs/FINAL_PROJECT_V2.md`에 있습니다.


In [ ]:
import os, sys, json, platform, subprocess, shutil, time, uuid
from pathlib import Path
IS_COLAB = platform.system() == 'Linux' and Path('/content').is_dir()
TEAM = 'team01'
SOURCE_ROOT = os.environ.get('FPV2_SOURCE', '')  # 업로드 후 실제 원격 vla_pipeline 경로
REPO_URL = 'https://github.com/semin126/VLA2026.git'
REPO_REF = 'main'                              # ver2.0이 push된 브랜치/commit
INSTALL = IS_COLAB and os.environ.get('FPV2_SKIP_INSTALL') != '1'
USE_DRIVE = IS_COLAB and os.environ.get('FPV2_NO_DRIVE') != '1'
PYTHON_OVERRIDE = os.environ.get('FPV2_PYTHON', '')
WORK = Path(os.environ.get('FPV2_WORK', '/content/Final_project_v2' if IS_COLAB else 'outputs/final_project_v2'))
SCENES_ROOT = os.environ.get('FPV2_SCENES', '')   # 같은 suite.json 재사용 위치
CUSTOM_SCENE = os.environ.get('FPV2_SCENE', '')    # 직접 저장한 씬 JSON. 비우면 개발 open
TASK_COMMAND = ''                              # 시작 명령. 비우면 scene.json의 command 사용
RUN_CONDITIONS = ['D']                         # A,B,C,D. E는 로컬 전용 셀
SECONDS = float(os.environ.get('FPV2_SECONDS', '12'))  # 연결 확인용. 연구 시 예비시험으로 결정
MAX_STEPS = int(os.environ.get('FPV2_MAX_STEPS', '2400'))
ACTION_STEPS = 240
OBSERVER_STEPS = 60
MAX_CALLS = 8
REPEAT_SEEDS = '42'
VLA_SCENE = ''                                 # A: 전체 과제 명령을 학습한 모델
VLA_SUBTASK = ''                               # B/C: 부분 운반 지시를 학습한 같은 모델
VLM_MODEL = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
STRATEGY_OVERRIDE = os.environ.get('FPV2_STRATEGY', '')
B_STRATEGY_OVERRIDE = os.environ.get('FPV2_B_STRATEGY', '')
FINAL_TEST = False                            # 전략·모델 확정 후에만 True


## 코드·실행 환경·저장 위치

아직 push하지 않은 코드는 `Upload to Colab` 후 `SOURCE_ROOT`로 지정합니다. 기존 작업 폴더를 자동 pull/reset하지 않습니다. Linux에서는 독립 Python 3.12 환경과 기존 잠금 파일을 사용합니다.

Drive는 각 명령 종료 시 해시를 확인하여 백업합니다. 세션이 도중에 사라지면 미복사 파일은 잃을 수 있습니다. 모델 로딩 시간은 과제 시간과 별도로 기록합니다.


In [ ]:
if SOURCE_ROOT:
    PROJECT = Path(SOURCE_ROOT).expanduser().resolve()
else:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base, base/'vla_pipeline', base/'2026/VLA2026/vla_pipeline']
    PROJECT = next((p for p in candidates if (p/'scripts/final_project_v2.py').is_file()), None)
    if PROJECT is None:
        checkout = Path('/content' if IS_COLAB else '.') / ('fpv2_checkout_'+uuid.uuid4().hex[:8])
        subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', REPO_REF], check=True)
        PROJECT = checkout/'vla_pipeline'
PROJECT = PROJECT.resolve()
if not (PROJECT/'vlapipe/final_project_v2/runner.py').is_file():
    raise RuntimeError('ver2.0 코드가 없습니다. 최신 브랜치 또는 업로드 위치를 확인하세요.')
sys.path.insert(0, str(PROJECT))
from vlapipe.final_project_v2.notebook_runtime import NotebookRun, prepare_python
if not WORK.is_absolute():
    WORK = PROJECT/WORK
if USE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/VLA2026/Final_project_v2')
else:
    BACKUP = None
run = NotebookRun(PROJECT, WORK, BACKUP, TEAM)
PY = prepare_python(PROJECT, WORK, install=INSTALL, python_override=PYTHON_OVERRIDE)
run.python = PY
B_STRATEGY = Path(B_STRATEGY_OVERRIDE) if B_STRATEGY_OVERRIDE else PROJECT/'final_project_v2/baseline_strategy.py'
C_STRATEGY = Path(STRATEGY_OVERRIDE) if STRATEGY_OVERRIDE else PROJECT/'final_project_v2/student_strategy.py'
DATA_STRATEGY = B_STRATEGY  # 같은 TXT 전략으로 초기 VLM 학습 후보 생성
print('PROJECT:', PROJECT, '\nPYTHON:', PY, '\nRUN:', run.root, '\nBACKUP:', run.backup)
run.run('scripts/check_env.py')


In [ ]:
if SCENES_ROOT:
    SCENES = Path(SCENES_ROOT).resolve()
else:
    SCENES = run.root/'scenes'
    run.run('scripts/final_project_v2.py', 'scenes', '--out', SCENES, '--tests', 10)
registry = json.loads((SCENES/'suite.json').read_text(encoding='utf-8'))
DEV_SCENES = [SCENES/r['path'] for r in registry['scenes'] if r['split']=='development']
print('Scenes:', SCENES, '\nDevelopment:', *DEV_SCENES, sep='\n')
from IPython.display import display, Image, HTML
import base64
def show_video(path):
    encoded = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    display(HTML('<video controls width="700"><source src="data:video/mp4;base64,'+encoded+'" type="video/mp4"></video>'))
display(Image(filename=str(DEV_SCENES[0].with_suffix('')/'scene.png'), width=700))


## 추가 데이터 생성

`train / validation / sft_test` 혼합 씬만 사용합니다. 최종 비교용 `test`와 개발용 `development`는 거절합니다. sft_test는 학습 데이터의 독립 평가용이며 최종 로봇 시험과 다릅니다.

VLM: 이미지＋전략＋실행 문맥 → 검사·계획 JSON. VLA: 이미지 2종＋상태16＋명령 → 실제 action7. VLM 응답만으로 VLA 동작 데이터를 만들 수 없습니다. 생성 응답은 pending 후보이며 사람이 검토해야 정답 데이터가 됩니다.


In [ ]:
GENERATE_CANDIDATES = False
CANDIDATES = run.root/'vlm_candidates'
if GENERATE_CANDIDATES:
    args = ['candidates', '--scenes', SCENES,
            '--strategy', DATA_STRATEGY, '--model', VLM_MODEL, '--out', CANDIDATES]
    if TASK_COMMAND.strip():
        args += ['--task-command', TASK_COMMAND]
    run.run('scripts/final_project_v2.py', *args)
    print('이미지·candidates.json 검토 후 review.json에 accepted/rejected, reviewer, 응답 수정을 기록하세요.')


## 피드백 후보

초기 이미지 데이터만으로 실패 후 재계획을 학습할 수는 없습니다. 학습용 씬에서 C를 실행한 뒤 실제 이미지·프롬프트·이력을 가져옵니다. 마지막 성공만 보고 중간 계획을 모두 정답 처리하지 않습니다.


In [ ]:
IMPORT_FEEDBACK = False
FEEDBACK_TRIAL_ROOT = ''
FEEDBACK_CANDIDATES = run.root/'feedback_candidates'
if IMPORT_FEEDBACK:
    run.run('scripts/final_project_v2.py', 'feedback-candidates', '--root', FEEDBACK_TRIAL_ROOT,
            '--out', FEEDBACK_CANDIDATES)


In [ ]:
EXPORT_REVIEWED = False
VLM_DATA = run.root/'vlm_reviewed'
REVIEWED_DIRS = [CANDIDATES]
if EXPORT_REVIEWED:
    args = ['export-reviewed', '--out', VLM_DATA]
    for folder in REVIEWED_DIRS:
        args += ['--candidates', folder]
    run.run('scripts/final_project_v2.py', *args)
    run.run('scripts/finetune_planner.py', '--dataset', VLM_DATA, '--out', run.root/'vlm_dry_run', '--dry-run')


## VLA의 실제 동작 데이터

D-state를 실제 실행하고 같은 궤적에서 전체 과제 라벨(A)과 부분 지시 라벨(B/C)을 각각 저장합니다. 전체는 완료된 씬, 부분은 실제로 들어서 올바르게 놓은 구간만 accepted입니다. 실패도 rejected로 남깁니다. 원본 seed/family로 분할한 뒤 변환합니다. 고전 제어 데이터이며 사람 시연으로 표기하지 않습니다.


In [ ]:
COLLECT_ACTIONS = False
COLLECTIONS = run.root/'vla_collections'
if COLLECT_ACTIONS:
    for row in registry['scenes']:
        if row['split'] not in {'train','validation','sft_test'}:
            continue
        args = ['collect-actions', '--scene', SCENES/row['path'],
                '--out', COLLECTIONS/Path(row['path']).stem, '--seconds', 120]
        if TASK_COMMAND.strip():
            args += ['--task-command', TASK_COMMAND]
        run.run('scripts/final_project_v2.py', *args)


In [ ]:
PREPARE_VLA = False
if PREPARE_VLA:
    for system, label in [('single','scene'), ('hier','subtask')]:
        run.run('scripts/final_project_v2.py', 'prepare-actions', '--collections', COLLECTIONS,
                '--system', system, '--out', run.root/('vla_'+system),
                '--repo-id', 'dshs2026/final_project_v2_'+system, '--convert')


## 학습과 저장 모델 비교

1스텝은 연결 검사입니다. 학습량은 validation에서 정하고 최종 시험에서 다시 고르지 않습니다. A는 single, B/C는 hier 모델을 쓰며 원본 궤적·업데이트 수·데이터량 차이를 기록합니다. B/C는 같은 체크포인트·전략을 씁니다.


In [ ]:
TRAIN_VLM = False
TRAIN_VLA = False
VLM_STEPS = 1
VLA_STEPS = 2
if TRAIN_VLM:
    run.run('scripts/finetune_planner.py', '--dataset', VLM_DATA, '--model', VLM_MODEL,
            '--out', run.root/'vlm_train', '--max-steps', VLM_STEPS, '--gradient-checkpointing')
if TRAIN_VLA:
    for system in ('single','hier'):
        run.run('scripts/train_vla.py', '--dataset-root', run.root/('vla_'+system)/'lerobot',
                '--repo-id', 'dshs2026/final_project_v2_'+system, '--out', run.root/('vla_train_'+system),
                '--steps', VLA_STEPS, '--batch-size', 1)
run.sync()
print('Artifacts:', run.root, '\nBackup:', run.backup)
